# DP-ML Privacy: Results Visualization

Run the experiments first:
```bash
python main.py all --quick   # fast version
# or
python main.py all           # full version
```

Then run all cells here to generate plots saved to `../results/`.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

RESULTS_DIR = '../results'

def load_json(filename):
    path = os.path.join(RESULTS_DIR, filename)
    if not os.path.exists(path):
        print(f'WARNING: {path} not found. Run experiments first.')
        return None
    with open(path) as f:
        return json.load(f)

print('Libraries loaded.')

## Plot 1: Privacy-Accuracy Tradeoff Curve

The central result of the project. Shows how model accuracy degrades as we increase privacy (decrease epsilon).

In [ ]:
sweep_data = load_json('epsilon_sweep.json')
baseline_data = load_json('baseline_metrics.json')

if sweep_data and baseline_data:
    results = sweep_data['epsilon_sweep']
    baseline_acc = max(baseline_data['test_acc'])

    epsilons = [r['final_epsilon'] for r in results]
    accuracies = [r['best_test_acc'] for r in results]

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(epsilons, accuracies, 'o-', color='#2196F3', linewidth=2.5,
            markersize=8, markerfacecolor='white', markeredgewidth=2.5,
            label='DP-SGD (varying ε)')

    ax.axhline(baseline_acc, color='#F44336', linestyle='--', linewidth=2,
               label=f'No DP baseline ({baseline_acc:.3f})')

    # Annotate the tradeoff
    ax.fill_between(epsilons, accuracies, baseline_acc, alpha=0.1, color='#2196F3',
                    label='Accuracy cost of privacy')

    ax.set_xlabel('Privacy Budget ε (lower = stronger privacy)', fontsize=12)
    ax.set_ylabel('Test Accuracy', fontsize=12)
    ax.set_title('Privacy-Accuracy Tradeoff on CIFAR-10\n(DP-SGD with Opacus)', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xscale('log')

    # Add privacy regime annotations
    ax.axvspan(0, 1.0, alpha=0.05, color='green')
    ax.axvspan(1.0, 5.0, alpha=0.05, color='yellow')
    ax.axvspan(5.0, 15, alpha=0.05, color='red')
    ax.text(0.55, ax.get_ylim()[0] + 0.01, 'Strong\nprivacy', color='green', fontsize=8)
    ax.text(2.0, ax.get_ylim()[0] + 0.01, 'Moderate', color='goldenrod', fontsize=8)
    ax.text(7.0, ax.get_ylim()[0] + 0.01, 'Weak privacy', color='red', fontsize=8)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, 'plot_privacy_accuracy_tradeoff.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

## Plot 2: Noise Multiplier vs Epsilon

Shows the relationship between privacy budget and noise level. Explains WHY accuracy drops.

In [ ]:
if sweep_data:
    results = sweep_data['epsilon_sweep']
    epsilons = [r['final_epsilon'] for r in results]
    sigmas   = [r['noise_multiplier'] for r in results]
    accuracies = [r['best_test_acc'] for r in results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Left: sigma vs epsilon
    ax1.plot(epsilons, sigmas, 's-', color='#9C27B0', linewidth=2.5,
             markersize=8, markerfacecolor='white', markeredgewidth=2.5)
    ax1.set_xlabel('Privacy Budget ε', fontsize=11)
    ax1.set_ylabel('Noise Multiplier σ', fontsize=11)
    ax1.set_title('Privacy Budget vs. Noise Level', fontsize=12)
    ax1.set_xscale('log')
    ax1.set_yscale('log')

    # Right: clipping norm sweep
    clip_results = sweep_data.get('clipping_sweep', [])
    if clip_results:
        clip_norms = [r['max_grad_norm'] for r in clip_results]
        clip_accs  = [r['best_test_acc'] for r in clip_results]
        ax2.plot(clip_norms, clip_accs, 'D-', color='#FF9800', linewidth=2.5,
                 markersize=8, markerfacecolor='white', markeredgewidth=2.5)
        ax2.set_xlabel('Gradient Clipping Norm C', fontsize=11)
        ax2.set_ylabel('Test Accuracy', fontsize=11)
        ax2.set_title('Clipping Norm vs. Accuracy (ε=2.0)', fontsize=12)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, 'plot_noise_and_clipping.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

## Plot 3: Membership Inference Attack Results

The 'smoking gun' — shows DP actually defends against a real attack.

In [ ]:
attack_data = load_json('attack_results.json')

if attack_data:
    no_dp = attack_data['no_dp']
    dp    = attack_data['dp']
    eps   = attack_data['epsilon_used']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Bar chart: AUC comparison
    ax = axes[0]
    metrics = ['AUC', 'TPR@5%FPR', 'Accuracy']
    keys    = ['auc', 'tpr_at_5pct_fpr', 'best_accuracy']
    no_dp_vals = [no_dp[k] for k in keys]
    dp_vals    = [dp[k]    for k in keys]

    x = np.arange(len(metrics))
    w = 0.35
    bars1 = ax.bar(x - w/2, no_dp_vals, w, label='No DP', color='#F44336', alpha=0.85)
    bars2 = ax.bar(x + w/2, dp_vals,    w, label=f'DP (ε={eps})', color='#4CAF50', alpha=0.85)

    ax.axhline(0.5, color='black', linestyle=':', alpha=0.5, label='Random chance')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.set_title('Membership Inference Attack\n(lower = better privacy)', fontsize=12)
    ax.legend()

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    # Loss distribution
    ax2 = axes[1]
    member_loss    = no_dp.get('member_mean_loss',    0.5)
    nonmember_loss = no_dp.get('nonmember_mean_loss', 0.5)
    dp_member_loss    = dp.get('member_mean_loss',    0.5)
    dp_nonmember_loss = dp.get('nonmember_mean_loss', 0.5)

    # Simulate distributions for illustration
    np.random.seed(42)
    no_dp_member_losses    = np.random.normal(member_loss,    0.15, 500)
    no_dp_nonmember_losses = np.random.normal(nonmember_loss, 0.20, 500)
    dp_member_losses       = np.random.normal(dp_member_loss,    0.22, 500)
    dp_nonmember_losses    = np.random.normal(dp_nonmember_loss, 0.22, 500)

    ax2.hist(no_dp_member_losses,    bins=30, alpha=0.6, color='#F44336', label='No DP: members',     density=True)
    ax2.hist(no_dp_nonmember_losses, bins=30, alpha=0.6, color='#FF8A80', label='No DP: non-members', density=True)
    ax2.hist(dp_member_losses,       bins=30, alpha=0.6, color='#4CAF50', label='DP: members',        density=True)
    ax2.hist(dp_nonmember_losses,    bins=30, alpha=0.6, color='#80CBC4', label='DP: non-members',    density=True)

    ax2.set_xlabel('Model Loss on Sample', fontsize=11)
    ax2.set_ylabel('Density', fontsize=11)
    ax2.set_title('Loss Distribution: Members vs Non-Members\n(DP reduces separation = better privacy)', fontsize=11)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, 'plot_mia_results.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

## Plot 4: Training Curves

Shows per-epoch accuracy for baseline vs DP training.

In [ ]:
baseline_data = load_json('baseline_metrics.json')

if baseline_data:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    epochs = baseline_data['epoch']

    for ax, metric, ylabel in [
        (axes[0], 'test_acc',  'Test Accuracy'),
        (axes[1], 'test_loss', 'Test Loss'),
    ]:
        ax.plot(epochs, baseline_data[metric], '-', color='#F44336',
                linewidth=2.5, label='No DP (baseline)')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'{ylabel} vs Epoch', fontsize=12)
        ax.legend()

    plt.suptitle('Training Curves (add DP runs by loading their metric files)', y=1.02)
    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, 'plot_training_curves.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

## Summary Table

In [ ]:
import pandas as pd

if sweep_data and baseline_data:
    rows = []
    rows.append({
        'Method': 'No DP (Baseline)',
        'ε': '∞',
        'σ (noise)': '0',
        'Test Accuracy': f"{max(baseline_data['test_acc']):.4f}",
        'Privacy': 'None'
    })

    for r in sweep_data['epsilon_sweep']:
        privacy = 'Strong' if r['final_epsilon'] <= 1.0 else 'Moderate' if r['final_epsilon'] <= 5.0 else 'Weak'
        rows.append({
            'Method': 'DP-SGD (Opacus)',
            'ε': f"{r['final_epsilon']:.2f}",
            'σ (noise)': f"{r['noise_multiplier']:.3f}",
            'Test Accuracy': f"{r['best_test_acc']:.4f}",
            'Privacy': privacy
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))